# Lineární regrese: prvních 5 minut → dalších 10 minut

Spusťte buňky shora dolů. Připravte alespoň čtyři dny minutových CSV se sloupci
`Timestamp`, `Čas`, `ODCHYLKA ČR [MW]` z období bez změny letního a zimního času.

Model se učí z úplných historických čtvrthodin a používá běžnou lineární regresi
bez penalizace. Pro každý testovací den se učí pouze ze starších dnů.
Příklad: z `12:00–12:04` odhaduje `12:05–12:14`.
CSV není potřeba nahrávat do veřejného repozitáře.


## 1. Nainstalovat závislosti


In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn


## 2. Stáhnout skript z GitHubu


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "000307/Odchylky"
BRANCH = "main"
script = Path("predikce_linearni_regrese.py")
urlretrieve(
    f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}/{script.name}",
    script,
)
compile(script.read_text(encoding="utf-8"), str(script), "exec")
print(f"Stažen skript: {script}")


## 3. Nahrát všechna vstupní CSV


In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = sorted(name for name in uploaded if name.lower().endswith(".csv"))
if not csv_files:
    raise ValueError("Nahrajte vstupní CSV s minutovými odchylkami.")
print(f"Vybráno CSV souborů: {len(csv_files)}")


## 4. Vyhodnotit model a zobrazit graf

Správná polarita znamená znaménko **průměru** dalších deseti minut,
nikoli stejné znaménko v každé z nich. Test na několika dnech sám o sobě
neprokazuje dlouhodobou přesnost.


In [ ]:
import os
import subprocess
import sys
from IPython.display import Image, display

out_dir = Path("predikce_lr")
subprocess.run(
    [sys.executable, str(script), *csv_files, "--out", str(out_dir)],
    check=True,
    env={**os.environ, "MPLBACKEND": "Agg"},
)
display(Image(filename=str(out_dir / "predikce.png")))


## 5. Stáhnout výsledky

ZIP obsahuje tabulky, souhrn a graf.


In [ ]:
from shutil import make_archive

archive = make_archive("vysledky_linearni_regrese", "zip", root_dir=out_dir)
files.download(archive)


## Volitelně: předpověď nového intervalu

Po získání prvních pěti minut nahraďte ukázkový začátek a všech pět hodnot.
Potom nastavte `SPUSTIT_PREDPOVED = True` a spusťte buňku.
K učení se použijí pouze úplné intervaly ukončené nejpozději před začátkem
předpovídané čtvrthodiny. Výstup obsahuje deset minutových odhadů a souhrn
včetně odhadu průměru celé čtvrthodiny.


In [ ]:
SPUSTIT_PREDPOVED = False
START = "2026-09-29 12:00"
PRVNICH_PET_MW = [10, 20, 30, 40, 50]  # Nahraďte skutečnými hodnotami.

if SPUSTIT_PREDPOVED:
    forecast_dir = Path("nova_predikce_lr")
    subprocess.run(
        [sys.executable, str(script), *csv_files,
         "--first-five", *map(str, PRVNICH_PET_MW),
         "--start", START, "--out", str(forecast_dir)],
        check=True,
        env={**os.environ, "MPLBACKEND": "Agg"},
    )
    import pandas as pd
    display(pd.read_csv(forecast_dir / "predikce_noveho_intervalu.csv"))
    display(pd.read_csv(forecast_dir / "souhrn_predikce.csv"))
    forecast_archive = make_archive("nova_predikce_linearni_regrese", "zip", root_dir=forecast_dir)
    files.download(forecast_archive)
